In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torchvision.transforms import v2

sys.path.append(os.path.abspath(".."))
from src.architectures import AlexNetCIFAR, GeneralMLP
from src.continual_learning import GPM, CLMetricsTracker
from src.utils import (
    apply_heavy_tailed_init,
    apply_heavy_tailed_init_class_il,
    apply_heavy_tailed_init_fc_only,
    save_snapshot,
    set_seed,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 1. GPU-Accelerated Data Loading ---
def generate_permutations(num_tasks, num_pixels=784, seed=42):
    """Generates clean pseudorandom domain permutations mapping each task stream."""
    rng = np.random.RandomState(seed)
    perms = [
        torch.arange(num_pixels).to(DEVICE)
    ]  # Task 0 is standard un-permuted MNIST
    for _ in range(num_tasks - 1):
        perms.append(torch.from_numpy(rng.permutation(num_pixels)).to(DEVICE))
    return perms


def get_gpu_data(dataset):
    """Loads raw tensors to GPU once to avoid repetitive overhead."""
    imgs = torch.stack([img for img, _ in dataset]).to(DEVICE).view(-1, 784)
    lbls = torch.tensor([lbl for _, lbl in dataset]).to(DEVICE)
    return imgs, lbls


def get_split_cifar100_tasks(seed):
    rng = np.random.RandomState(seed)
    class_order = rng.permutation(100)
    task_data = []

    for t in range(NUM_TASKS):
        task_classes = class_order[t * CLASSES_PER_TASK : (t + 1) * CLASSES_PER_TASK]
        task_classes_tensor = torch.tensor(task_classes, device=DEVICE)

        tx = x_train_all[torch.isin(y_train_all, task_classes_tensor)]
        ty = y_train_all[torch.isin(y_train_all, task_classes_tensor)]
        te_x = x_test_all[torch.isin(y_test_all, task_classes_tensor)]
        te_y = y_test_all[torch.isin(y_test_all, task_classes_tensor)]

        task_data.append(
            {
                "classes": task_classes,
                "train_x": tx,
                "train_y": ty,
                "test_x": te_x,
                "test_y": te_y,
            }
        )
    return task_data


def evaluate_task_stream(model, tasks_data, max_task_idx):
    """Evaluates task-incremental accuracy across seen tasks cleanly without batching artifacts."""
    model.eval()
    task_accuracies = []

    with torch.no_grad():
        for t_idx in range(max_task_idx + 1):
            te_x = tasks_data[t_idx]["test_x"]
            te_y = tasks_data[t_idx]["test_y"]
            task_classes = tasks_data[t_idx]["classes"]
            task_classes_tensor = torch.tensor(task_classes, device=DEVICE)

            # Split CIFAR-100 test set per task is 500 images; evaluate in one forward pass on GPU
            logits = model(te_x)

            # Restrict predictions strictly to the current task's output subspace
            task_logits = logits[:, task_classes]
            local_preds = task_logits.argmax(dim=1)
            global_preds = task_classes_tensor[local_preds]

            correct = (global_preds == te_y).sum().item()
            total = te_y.size(0)

            task_accuracies.append(correct / total)

    return task_accuracies


# --- 1. FULL-STREAM EVALUATION (Tracks All 20 Tasks for Zero-Shot & Retention) ---
def evaluate_full_stream(model, tasks_data):
    """Evaluates task-incremental accuracy across ALL 20 tasks (past, current, future)."""
    model.eval()
    task_accuracies = []

    with torch.no_grad():
        for t_idx in range(len(tasks_data)):
            te_x = tasks_data[t_idx]["test_x"]
            te_y = tasks_data[t_idx]["test_y"]
            task_classes = tasks_data[t_idx]["classes"]
            task_classes_tensor = torch.tensor(task_classes, device=DEVICE)

            logits = model(te_x)

            # Isolated task-incremental evaluation
            task_logits = logits[:, task_classes]
            local_preds = task_logits.argmax(dim=1)
            global_preds = task_classes_tensor[local_preds]

            correct = (global_preds == te_y).sum().item()
            task_accuracies.append(correct / te_y.size(0))

    return (
        task_accuracies  # Returns list of length 20: [R_{curr, 0}, ..., R_{curr, 19}]
    )

# **MLP + P-MNIST**

In [ ]:
# --- 1. EXPERIMENTAL CONFIGURATION ---
RUN_SEEDS = [0]
NUM_TASKS = 20
EPOCHS_PER_TASK = 5
LR_TASK_0 = 1e-3
LR_SUBSEQUENT = 1e-3
BATCH_SIZE = 256
CALIB_SAMPLE_SIZE = 1024

ALPHA_INIT = 2.0
G_INIT = 1.0

# Structural variables matching target deep architecture
hidden_size = 784
depth = 9
activation_name = "tanh"
bias = False

# Algorithm Configuration
GLOBAL_THRESHOLD = 0.97
EVAL_EVERY_N_BATCHES = 100

# --- 2. OPTIMIZED DATA LOADING ---
print(f"Initializing VRAM memory pinning pipeline on device: {DEVICE}")
mnist_train = datasets.MNIST(
    "../data", train=True, download=True, transform=transforms.ToTensor()
)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)

train_imgs, train_lbls = get_gpu_data(mnist_train)
test_imgs_raw, test_lbls = get_gpu_data(mnist_test)


# --- 3. CORE BACKGROUND SWEEP PIPELINE ---
for current_seed in RUN_SEEDS:
    print("\n" + "=" * 80)
    print(f"LAUNCHING PARAMETER SWEEP FOR RANDOM EXPERIMENTAL SEED [s={current_seed}]")
    print("=" * 80)

    set_seed(current_seed)
    task_permutations = generate_permutations(num_tasks=NUM_TASKS, seed=current_seed)

    model = GeneralMLP(784, hidden_size, 10, depth, activation_name, bias=bias).to(
        DEVICE
    )
    model = apply_heavy_tailed_init(
        model=model, alpha=ALPHA_INIT, g=G_INIT, seed=current_seed
    )

    optimizer = optim.SGD(model.parameters(), lr=LR_TASK_0)
    criterion = nn.CrossEntropyLoss()
    tracker = CLMetricsTracker(max_tasks=NUM_TASKS)
    gpm = GPM(variance_threshold=GLOBAL_THRESHOLD)

    linear_layer_info = [
        (name + ".weight", module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
    ]

    total_steps = 0

    print(f"Architecture Depth: {depth} layers | Hidden Width: {hidden_size} neurons")
    print(f"Target Subspace Energy: {GLOBAL_THRESHOLD * 100:.1f}%")
    print("-" * 80)

    # --- 3.1 INITIALIZATION SNAPSHOT (T=0 / PRE-TRAINING BASELINE) ---
    model.eval()
    init_calib_indices = torch.randperm(len(train_imgs), device=DEVICE)[
        :CALIB_SAMPLE_SIZE
    ]
    # Pass Task 1 input distribution through initialized weights to record R_0
    init_calib_images = train_imgs[init_calib_indices][:, task_permutations[0]]

    with torch.no_grad():
        init_layer_inputs = model.get_layer_inputs(init_calib_images)

    snapshot_init = save_snapshot(
        model=model,
        gpm=gpm,
        layer_inputs_dict=init_layer_inputs,
        output_dir="./checkpoints/mlp",
        t_idx=-1,  # Generates snapshot_T00_s{seed}.pt
        alpha=ALPHA_INIT,
        g=G_INIT,
        seed=current_seed,
    )
    print(f"[INITIALIZATION SNAPSHOT CAPTURED] Saved to: {snapshot_init.name}")

    # --- 4. MAIN CONTINUAL LEARNING STREAM ---
    for t_idx in range(NUM_TASKS):
        current_lr = LR_TASK_0 if t_idx == 0 else LR_SUBSEQUENT
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        current_perm = task_permutations[t_idx]
        tx = train_imgs[:, current_perm]
        ty = train_lbls

        print(
            f"\n--- Task {t_idx:02d}/{NUM_TASKS:02d} | Seed: {current_seed} | Active LR: {current_lr:.1e} ---"
        )

        for epoch in range(EPOCHS_PER_TASK):
            model.train()
            indices = torch.randperm(len(tx), device=DEVICE)

            for i in range(0, len(tx), BATCH_SIZE):
                batch_idx = indices[i : i + BATCH_SIZE]
                bx, by = tx[batch_idx], ty[batch_idx]

                optimizer.zero_grad(set_to_none=True)

                output = model(bx)
                loss_current = criterion(output, by)
                loss_current.backward()

                gpm.project_model_gradients(model)
                optimizer.step()

                if total_steps % EVAL_EVERY_N_BATCHES == 0:
                    current_accs = []
                    model.eval()
                    with torch.no_grad():
                        for eval_t_idx in range(t_idx + 1):
                            eval_perm = task_permutations[eval_t_idx]
                            test_x = test_imgs_raw[:, eval_perm]

                            outputs = model(test_x)
                            preds = outputs.argmax(dim=1)
                            acc = (preds == test_lbls).float().mean().item()
                            current_accs.append(acc)

                    tracker.log(
                        step=total_steps,
                        acc_list=current_accs,
                    )

                    acc_report = " | ".join(
                        [
                            f"T{j}: {current_accs[j] * 100:.1f}%"
                            for j in range(t_idx + 1)
                        ]
                    )
                    print(
                        f"Seed {current_seed} | Step {total_steps:04d} (Epoch {epoch}) -> {acc_report}"
                    )

                    model.train()

                total_steps += 1

        # --- 5. POST-TASK CALIBRATION STEP & END-OF-TASK LOGGING ---
        model.eval()
        calib_indices = torch.randperm(len(tx), device=DEVICE)[:CALIB_SAMPLE_SIZE]
        calib_images = tx[calib_indices]

        with torch.no_grad():
            layer_inputs_dict = model.get_layer_inputs(calib_images)

        var_total_dict = {}
        var_proj_dict = {}
        var_res_dict = {}

        for (weight_param_name, module), (_, layer_input) in zip(
            linear_layer_info, layer_inputs_dict.items()
        ):
            added_rank, diag = gpm.update_basis(
                layer_id=weight_param_name,
                module=module,
                live_activations=layer_input,
                threshold=GLOBAL_THRESHOLD,
            )

            key = weight_param_name.replace(".", "_")
            var_total_dict[key] = diag["total_variance_sq"]
            var_proj_dict[key] = diag["norm_projected_sq"]
            var_res_dict[key] = diag["residual_variance_sq"]

        snapshot_file = save_snapshot(
            model=model,
            gpm=gpm,
            layer_inputs_dict=layer_inputs_dict,
            output_dir="./checkpoints/mlp",
            t_idx=t_idx,
            alpha=ALPHA_INIT,
            g=G_INIT,
            seed=current_seed,
        )

        post_task_accs = []
        with torch.no_grad():
            for eval_t_idx in range(t_idx + 1):
                eval_perm = task_permutations[eval_t_idx]
                test_x = test_imgs_raw[:, eval_perm]

                outputs = model(test_x)
                preds = outputs.argmax(dim=1)
                acc = (preds == test_lbls).float().mean().item()
                post_task_accs.append(acc)

        tracker.log(
            step=total_steps,
            acc_list=post_task_accs,
            task_idx=t_idx,
            basis_rank=gpm.get_basis_ranks(),
            total_basis_rank=gpm.get_total_basis_rank(),
            var_total=var_total_dict,
            var_proj=var_proj_dict,
            var_res=var_res_dict,
        )

        print(
            f"Task {t_idx} complete. Subspace memory updated. "
            f"Current total basis rank: {gpm.get_total_basis_rank()}"
        )

    # --- 6. END-OF-TRAINING SERIALIZATION ---
    output_dir = Path("./checkpoints/mlp")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_filename = output_dir / f"gpm_a{ALPHA_INIT}_run_s{current_seed}.csv"
    tracker.save_to_csv(output_filename)
    print(
        f"Sweep for seed {current_seed} serialized successfully to {output_filename}.\n"
    )

print("\n" + "=" * 80)
print("ALL MULTI-SEED PARAMETER CONVERSIONS COMPLETED IN BACKGROUND POOL.")
print("=" * 80)

# **AlexNet + Split CIFAR-100**

In [ ]:
# --- 1. EXPERIMENTAL CONFIGURATION ---
RUN_SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
NUM_TASKS = 10
CLASSES_PER_TASK = 10  # 20 tasks x 5 classes = 100 classes
EPOCHS_PER_TASK = 50
BATCH_SIZE = 64
CALIB_SAMPLE_SIZE = 1024

LR_TASK_0 = 1e-3
LR_SUBSEQUENT = 1e-3
MOMENTUM = 0.0
WEIGHT_DECAY = 0.0

ALPHA_INIT = 1.2
G_INIT = 1.0

# GPM Projection Settings
GLOBAL_THRESHOLD = 0.97
EVAL_BATCH_SIZE = 500

# --- 2. OPTIMIZED VRAM DATA LOADING & TASK PARTITIONING ---
print(f"Preloading CIFAR-100 dataset into VRAM ({DEVICE})...")
train_raw = datasets.CIFAR100(root="../data", train=True, download=True)
test_raw = datasets.CIFAR100(root="../data", train=False, download=True)

x_train_all = (
    torch.tensor(train_raw.data, device=DEVICE).permute(0, 3, 1, 2).float() / 255.0
)
y_train_all = torch.tensor(train_raw.targets, device=DEVICE, dtype=torch.long)

x_test_all = (
    torch.tensor(test_raw.data, device=DEVICE).permute(0, 3, 1, 2).float() / 255.0
)
y_test_all = torch.tensor(test_raw.targets, device=DEVICE, dtype=torch.long)

cifar_mean = torch.tensor([0.5071, 0.4867, 0.4408], device=DEVICE).view(1, 3, 1, 1)
cifar_std = torch.tensor([0.2675, 0.2565, 0.2761], device=DEVICE).view(1, 3, 1, 1)
x_train_all = (x_train_all - cifar_mean) / cifar_std
x_test_all = (x_test_all - cifar_mean) / cifar_std

gpu_transforms = nn.Sequential(
    v2.RandomCrop(32, padding=4, padding_mode="reflect"),
    v2.RandomHorizontalFlip(p=0.5),
)


# --- 3. MAIN MULTI-SEED EXPERIMENT RUNNER ---
for current_seed in RUN_SEEDS:
    print("\n" + "=" * 80)
    print(
        f"LAUNCHING SPLIT CIFAR-100 GPM (ALEXNET) [Seed: {current_seed} | Alpha: {ALPHA_INIT} | Gain: {G_INIT}]"
    )
    print("=" * 80)

    set_seed(current_seed)
    tasks_stream = get_split_cifar100_tasks(seed=current_seed)

    # 5-Layer AlexNet backbone
    model = AlexNetCIFAR(
        num_classes=100, activation="tanh", bias=False, use_dropout=False
    ).to(DEVICE)
    model = apply_heavy_tailed_init_class_il(
        model=model, alpha=ALPHA_INIT, g=G_INIT, seed=current_seed
    )

    # for t in range(20):
    #     head_slice = model.classifier.weight[t * 5 : (t + 1) * 5]
    #     max_w = head_slice.abs().max().item()
    #     print(f"Task {t:02d} Head: Max |W_ij| = {max_w:.2f}")

    optimizer = optim.SGD(
        model.parameters(),
        lr=LR_TASK_0,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss()

    # --- INITIALIZE TWO DEDICATED TRACKERS ---
    # Tracker 1: Detailed milestone evaluation at the end of each task
    task_tracker = CLMetricsTracker(max_tasks=NUM_TASKS)

    # Tracker 2: Live per-epoch training dynamics and convergence curves
    epoch_tracker = CLMetricsTracker(max_tasks=NUM_TASKS)

    gpm = GPM(variance_threshold=GLOBAL_THRESHOLD)

    # Track only sequential backbone layers (excluding classifier head)
    learnable_layer_info = [
        (name, module)
        for name, module in model.named_modules()
        if name in ["conv1", "conv2", "conv3", "fc1", "fc2"]
    ]

    # --- TRAINING & EVALUATION LOOP WITH DUAL TRACKING ---
    total_steps = 0
    start_time = time.time()

    for t_idx in range(NUM_TASKS):
        current_lr = LR_TASK_0 if t_idx == 0 else LR_SUBSEQUENT
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        task = tasks_stream[t_idx]
        tx, ty = task["train_x"], task["train_y"]
        num_task_samples = tx.size(0)
        active_classes = task["classes"]
        num_complete_samples = (num_task_samples // BATCH_SIZE) * BATCH_SIZE

        print(
            f"\n--- Task {t_idx + 1:02d}/{NUM_TASKS:02d} (Classes: {active_classes}) | Active LR: {current_lr:.1e} ---"
        )

        for epoch in range(EPOCHS_PER_TASK):
            model.train()

            indices = torch.randperm(num_task_samples, device=DEVICE)
            running_loss = 0.0
            num_batches = 0

            for i in range(0, num_complete_samples, BATCH_SIZE):
                batch_idx = indices[i : i + BATCH_SIZE]
                bx, by = tx[batch_idx], ty[batch_idx]
                bx = gpu_transforms(bx)

                optimizer.zero_grad(set_to_none=True)
                output = model(bx)

                # Task-level logit masking
                mask = torch.full_like(output, fill_value=-1e9)
                mask[:, active_classes] = output[:, active_classes]

                loss = criterion(mask, by)
                loss.backward()

                gpm.project_model_gradients(model)
                optimizer.step()

                running_loss += loss.item()
                num_batches += 1
                total_steps += 1

            epoch_loss = running_loss / max(1, num_batches)

            # Evaluate immediate test accuracy on the active task
            model.eval()
            with torch.no_grad():
                cur_te_x = task["test_x"]
                cur_te_y = task["test_y"]
                task_classes_tensor = torch.tensor(active_classes, device=DEVICE)

                t_logits = model(cur_te_x)[:, active_classes]
                local_preds = t_logits.argmax(dim=1)
                global_preds = task_classes_tensor[local_preds]
                cur_acc = (global_preds == cur_te_y).float().mean().item()

            # 1. LIVE LOGGING: Track intra-task dynamics at every epoch
            epoch_tracker.log(
                step=total_steps,
                acc_list=[],
                task_idx=t_idx,
                epoch=epoch + 1,
                epoch_loss=epoch_loss,
                intra_task_acc=cur_acc,
            )

            if (epoch + 1) % 5 == 0 or epoch == EPOCHS_PER_TASK - 1:
                print(
                    f"Epoch [{epoch + 1:02d}/{EPOCHS_PER_TASK:02d}] | "
                    f"Loss: {epoch_loss:.4f} | "
                    f"Task {t_idx + 1} Current Acc: {cur_acc * 100:.2f}%"
                )

        # --- POST-TASK CALIBRATION & BASIS UPDATE ---
        model.eval()
        calib_idx = torch.randperm(num_task_samples, device=DEVICE)[:CALIB_SAMPLE_SIZE]
        calib_images = tx[calib_idx]

        with torch.no_grad():
            layer_inputs_dict = model.get_layer_inputs(calib_images)

        var_total_dict, var_proj_dict, var_res_dict = {}, {}, {}
        for module_name, module in learnable_layer_info:
            live_input = layer_inputs_dict[module_name]
            param_key = f"{module_name}.weight"
            added_rank, diag = gpm.update_basis(
                layer_id=param_key,
                module=module,
                live_activations=live_input,
                threshold=GLOBAL_THRESHOLD,
            )
            var_total_dict[module_name] = diag["total_variance_sq"]
            var_proj_dict[module_name] = diag["norm_projected_sq"]
            var_res_dict[module_name] = diag["residual_variance_sq"]

        # --- FULL 20-TASK EVALUATION MATRIX ---
        all_20_accs = evaluate_full_stream(model, tasks_stream)

        # 2. MILESTONE LOGGING: Track full-stream retention, zero-shot transfer, and basis rank
        task_tracker.log(
            step=total_steps,
            acc_list=all_20_accs,
            task_idx=t_idx,
            basis_rank=gpm.get_basis_ranks(),
            total_basis_rank=gpm.get_total_basis_rank(),
            var_total=var_total_dict,
            var_proj=var_proj_dict,
            var_res=var_res_dict,
        )

        seen_mean = np.mean(all_20_accs[: t_idx + 1])
        zero_shot_mean = (
            np.mean(all_20_accs[t_idx + 1 :]) if t_idx < NUM_TASKS - 1 else 0.0
        )

        print(
            f"[CALIBRATION COMPLETE] Task {t_idx + 1} Finished | "
            f"Retained Acc: {seen_mean * 100:.2f}% | "
            f"Zero-Shot Future Acc: {zero_shot_mean * 100:.2f}% | "
            f"Stored Rank: {gpm.get_total_basis_rank()}"
        )

    # --- EXPORT SEPARATE CSV FILES ---
    output_dir = Path("./checkpoints/alexnet_10_task")
    output_dir.mkdir(parents=True, exist_ok=True)

    # File 1: 20 rows (Post-task full evaluations & GPM rank metrics)
    task_tracker.save_to_csv(
        output_dir / f"gpm_alexnet_a{ALPHA_INIT}_s{current_seed}_tasks.csv"
    )

    # File 2: 600 rows (Epoch-by-epoch loss, AUC trajectories, convergence rates)
    epoch_tracker.save_to_csv(
        output_dir / f"gpm_alexnet_a{ALPHA_INIT}_s{current_seed}_curves.csv"
    )

print("\n" + "=" * 80)
print("BENCHMARK COMPLETED SUCCESSFULLY.")
print("=" * 80)